## Step 1: Settings and model

In [12]:
import time
import statistics
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# ---- Settings (change these to experiment) ----
MODEL_ID = "gpt2"   # Hugging Face model to benchmark
NEW_TOKENS = 200    # number of tokens to generate in each run
RUNS = 5            # runs per setting


# Use a GPU if one is available, otherwise run on the CPU
device = "cuda" if torch.cuda.is_available() else "cpu"


def sync():
    """Wait for all queued GPU work to finish.

    GPU calls return before the work is done, so without this the timer would
    stop too early. On the CPU there is nothing to wait for, so it does nothing.
    """
    if device == "cuda":
        torch.cuda.synchronize()


# Load the tokenizer and the model.
tok = AutoTokenizer.from_pretrained(MODEL_ID)

# .eval() switches the model to inference mode
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device).eval()
print(f"Tokenizer and model loaded for {MODEL_ID} on {device}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Tokenizer and model loaded for gpt2 on cpu


## Step 2: Helper functions

- `run(use_cache)` generates text once and returns the output and the time per token.
- `kv_cache_bytes(input_ids)` measures how much memory the KV cache takes for a given prompt.

In [13]:
def run(use_cache):
    """Generate NEW_TOKENS tokens once. Returns (output_ids, seconds_per_token).
    """
    sync()                                   # make sure no earlier GPU work is still running
    start = time.perf_counter()
    out = model.generate(**inputs,max_new_tokens=NEW_TOKENS,
        do_sample=False,use_cache=use_cache,
        pad_token_id=tok.eos_token_id)
    sync()
    total = time.perf_counter() - start
    n_new = out.shape[1] - prompt_len

    return out, total / n_new


def kv_cache_bytes(input_ids):
    """Measure the KV cache size in bytes after processing `input_ids` (the prompt).
    """
    with torch.no_grad():
        past = model(input_ids=input_ids, use_cache=True).past_key_values

    past = getattr(past, "self_attention_cache", past)
    if hasattr(past, "layers"):
        tensors = [t for layer in past.layers for t in (layer.keys, layer.values)]

    return sum(t.numel() * t.element_size() for t in tensors)

In [14]:
prompt = '''India is a country in south asia. It is the worlds 7th largest country by area.
The capital of India is New Delhi. There are 28 states as 8 union territories. T
he largest state in india is'''

inputs = tok(prompt, return_tensors="pt").to(device)   # text -> token ids, on the same device as the model
prompt_len = inputs["input_ids"].shape[1]              # number of tokens in the prompt
print(f"Prompt length: {prompt_len} tokens")

Prompt length: 48 tokens


In [15]:
print(f"model={MODEL_ID} | device={device} | dtype={next(model.parameters()).dtype} | "
      f"prompt={prompt_len} tokens | new={NEW_TOKENS} tokens | median of {RUNS} runs")

model=gpt2 | device=cpu | dtype=torch.float32 | prompt=48 tokens | new=200 tokens | median of 5 runs


In [16]:
for use_cache in (True, False):
    run(use_cache)

# Timed runs
per_token, outputs = {}, {}
for use_cache in (True, False):
    times = []
    for _ in range(RUNS):
        out, t = run(use_cache)
        times.append(t)
    outputs[use_cache] = out
    per_token[use_cache] = statistics.median(times)
    print(f"cache={use_cache}: {per_token[use_cache]*1000:.0f} ms/token")

# Compare the two settings
print(f"speedup: {per_token[False] / per_token[True]:.1f}x")
print("identical output:", torch.equal(outputs[True], outputs[False]))

# Memory formula: 2 x L x T x Hkv x D x B
cfg = model.config
L = cfg.num_hidden_layers                                           # number of transformer layers
Hkv = getattr(cfg, "num_key_value_heads", None) or cfg.num_attention_heads  # KV heads (GPT-2 has one per attention head)
D = getattr(cfg, "head_dim", None) or cfg.hidden_size // cfg.num_attention_heads  # head dimension
B = next(model.parameters()).element_size()                         # bytes per number (4 for FP32)

predicted = 2 * L * prompt_len * Hkv * D * B                        # T = prompt_len here
measured = kv_cache_bytes(inputs["input_ids"])
print(f"cache for {prompt_len} tokens | formula: {predicted:,} bytes | "
      f"measured: {measured:,} bytes | per token: {predicted // prompt_len:,}")

cache=True: 55 ms/token
cache=False: 639 ms/token
speedup: 11.7x
identical output: True
cache for 48 tokens | formula: 3,538,944 bytes | measured: 3,538,944 bytes | per token: 73,728
